# P4-2 · Bài 5 · Embedding và RAG

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q faiss-cpu sentence-transformers

## Lý thuyết

In [ ]:
# Cần mạng: tải model embedding bge-m3 (~2 GB) từ Hugging Face, chạy trên Colab
from sentence_transformers import SentenceTransformer
import faiss, numpy as np
emb = SentenceTransformer("BAAI/bge-m3")
X = emb.encode(chunks, normalize_embeddings=True)          # (n, d), đã chuẩn hóa độ dài 1
index = faiss.IndexFlatIP(X.shape[1]); index.add(X)        # tích vô hướng = cosine khi đã chuẩn hóa
q = emb.encode(["Chính sách đổi trả trong bao lâu?"], normalize_embeddings=True)
scores, ids = index.search(q, k=5)                         # 5 đoạn có cosine cao nhất

## Bài tập 4: Chunking quy chế có overlap {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
quy_che = ("Điều 12. Học bổng khuyến khích học tập. " + "Sinh viên được xét học bổng theo từng học kỳ dựa trên điểm trung bình và điểm rèn luyện. " * 3 +
           "Mức học bổng loại giỏi bằng 120% học phí của học kỳ đó, áp dụng khi điểm trung bình từ 3,2 và rèn luyện từ loại tốt. " +
           "Mức học bổng loại khá bằng 100% học phí. " + "Danh sách được công bố sau bốn tuần kể từ khi kết thúc học kỳ. " * 3)

def chunk(text, size, overlap, nguon):
    # TODO
    return []

In [ ]:
def chunk(text, size, overlap, nguon):
    words, step, out = text.split(), size - overlap, []
    for i in range(0, len(words), step):
        out.append({"id": f"{nguon}#{len(out)}", "nguon": nguon, "text": " ".join(words[i:i + size])})
        if i + size >= len(words):
            break
    return out

doan = chunk(quy_che, 40, 10, "quy_che_hoc_bong.pdf")
for d in doan:
    print(d["id"], "|", d["text"][:70], "...")

quy_che_hoc_bong.pdf#0 | Điều 12. Học bổng khuyến khích học tập. Sinh viên được xét học bổng th ...
quy_che_hoc_bong.pdf#1 | xét học bổng theo từng học kỳ dựa trên điểm trung bình và điểm rèn luy ...
quy_che_hoc_bong.pdf#2 | bình và điểm rèn luyện. Mức học bổng loại giỏi bằng 120% học phí của h ...
quy_che_hoc_bong.pdf#3 | loại tốt. Mức học bổng loại khá bằng 100% học phí. Danh sách được công ...
quy_che_hoc_bong.pdf#4 | bố sau bốn tuần kể từ khi kết thúc học kỳ. Danh sách được công bố sau  ...


## Bài tập 5: Tìm kiếm vector trên bộ FAQ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
faq = ["Thời gian giao hàng nội thành là 1 đến 2 ngày", "Đổi trả miễn phí trong 7 ngày nếu sản phẩm lỗi",
       "Thanh toán bằng chuyển khoản, thẻ hoặc tiền mặt khi nhận hàng", "Bàn trang điểm có ngăn kéo và gương LED",
       "Phí vận chuyển ngoại thành tính theo cân nặng", "Bảo hành khung bàn 12 tháng",
       "Có thể hủy đơn trước khi shop giao cho đơn vị vận chuyển", "Gương LED có ba chế độ ánh sáng"]
hoi = [("bao lâu thì giao hàng", 0), ("sản phẩm lỗi có được đổi trả không", 1), ("gương có mấy chế độ ánh sáng", 7),
       ("tôi muốn hủy đơn", 6), ("bảo hành bao lâu", 5), ("trả tiền khi nhận hàng được không", 2),
       ("mấy hôm thì tôi có đồ", 0)]
# TODO: vec = TfidfVectorizer().fit(faq); ma trận F, top-2 cho từng câu hỏi; recall2

In [ ]:
vec = TfidfVectorizer().fit(faq)
F = vec.transform(faq).toarray()
dung = 0
for cau, dap_an in hoi:
    q = vec.transform([cau]).toarray()[0]
    top = np.argsort(-(F @ q))[:2]
    dung += dap_an in top
    print(f"{cau:40s} → {list(top)} {'✓' if dap_an in top else '✗'}")
recall2 = dung / len(hoi)
print("recall@2 =", recall2)

bao lâu thì giao hàng                    → [0, 2] ✓
sản phẩm lỗi có được đổi trả không       → [1, 7] ✓
gương có mấy chế độ ánh sáng             → [7, 3] ✓
tôi muốn hủy đơn                         → [6, 0] ✓
bảo hành bao lâu                         → [5, 0] ✓
trả tiền khi nhận hàng được không        → [2, 1] ✓
mấy hôm thì tôi có đồ                    → [7, 3] ✗
recall@2 = 0.8571428571428571


## Bài tập 6: BM25 bắt được mã lỗi mà vector bỏ sót {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import math
docs = ["Lỗi E-401 xảy ra khi phiên đăng nhập hết hạn, hãy đăng nhập lại",
        "Thanh toán thất bại thường do thẻ hết hạn mức hoặc ngân hàng từ chối giao dịch",
        "Mã E-503 nghĩa là cổng thanh toán đang bảo trì, vui lòng thử lại sau 15 phút",
        "Hướng dẫn thanh toán bằng ví điện tử và thẻ ngân hàng",
        "Khi gặp lỗi thanh toán, kiểm tra kết nối mạng và thử lại"]
diem_vector = [0.21, 0.80, 0.74, 0.70, 0.78]        # model embedding "hiểu" ý thanh toán nhưng không coi trọng mã E-503
query = "thanh toán báo E-503"
DUNG = 2

def bm25(query, docs, k1=1.5, b=0.75):
    # TODO: trả về list điểm BM25 của từng tài liệu
    return [0.0] * len(docs)

In [ ]:
def bm25(query, docs, k1=1.5, b=0.75):
    toks = [d.lower().split() for d in docs]
    avg = sum(map(len, toks)) / len(toks); N = len(docs)
    diem = []
    for t in toks:
        s = 0.0
        for w in query.lower().split():
            n = sum(w in d for d in toks)
            idf = math.log((N - n + 0.5) / (n + 0.5) + 1)
            tf = t.count(w)
            s += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(t) / avg))
        diem.append(s)
    return diem

def xep_hang(diem):
    return sorted(range(len(diem)), key=lambda i: -diem[i])

xh_vector, xh_bm25 = xep_hang(diem_vector), xep_hang(bm25(query, docs))
rrf = [sum(1 / (60 + xh.index(i) + 1) for xh in (xh_vector, xh_bm25)) for i in range(len(docs))]
xh_rrf = xep_hang(rrf)
print("vector:", xh_vector, "| BM25:", xh_bm25, "| RRF:", xh_rrf)

vector: [1, 4, 2, 3, 0] | BM25: [2, 3, 1, 4, 0] | RRF: [1, 2, 3, 4, 0]


## Bài tập 7: Prompt có trích dẫn và kiểm tra trích dẫn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import re
doan = ["Đổi trả miễn phí trong 7 ngày nếu sản phẩm lỗi do nhà sản xuất.",
        "Khách tự đổi ý thì chịu phí vận chuyển hai chiều.",
        "Hàng giảm giá trên 50% không áp dụng đổi trả."]
tra_loi_mau = ["Bạn được đổi trả miễn phí trong 7 ngày nếu hàng lỗi [1]. Nếu chỉ đổi ý, bạn trả phí ship hai chiều [2].",
               "Bạn được đổi trả trong 30 ngày [4].",
               "Hàng giảm giá sâu không được đổi [3]. Shop luôn hoàn tiền trong 24 giờ."]

def build_rag_prompt(cau_hoi, doan):
    # TODO
    return ""

def kiem_tra(tra_loi, doan):
    # TODO: trả về (danh sách số trích dẫn không tồn tại, số câu không có trích dẫn)
    return [], 0

In [ ]:
def build_rag_prompt(cau_hoi, doan):
    nguon = "\n".join(f"[{i}] {d}" for i, d in enumerate(doan, 1))
    return (f"Trả lời câu hỏi CHỈ dựa trên các nguồn dưới đây. Sau mỗi câu, ghi số nguồn đã dùng như [1].\n"
            f"Nếu nguồn không đủ thông tin, trả lời: \"Tôi không tìm thấy thông tin này.\"\n\n<nguon>\n{nguon}\n</nguon>\n\n"
            f"Câu hỏi: {cau_hoi}")

def kiem_tra(tra_loi, doan):
    sai = [int(n) for n in re.findall(r"\[(\d+)\]", tra_loi) if not 1 <= int(n) <= len(doan)]
    cau = [c for c in re.split(r"(?<=[.!?])\s+", tra_loi.strip()) if c]
    khong_dan = sum(not re.search(r"\[\d+\]", c) for c in cau)
    return sai, khong_dan

print(build_rag_prompt("Tôi đổi ý thì có được trả hàng không?", doan))
for t in tra_loi_mau:
    print(kiem_tra(t, doan), "←", t)

Trả lời câu hỏi CHỈ dựa trên các nguồn dưới đây. Sau mỗi câu, ghi số nguồn đã dùng như [1].
Nếu nguồn không đủ thông tin, trả lời: "Tôi không tìm thấy thông tin này."

<nguon>
[1] Đổi trả miễn phí trong 7 ngày nếu sản phẩm lỗi do nhà sản xuất.
[2] Khách tự đổi ý thì chịu phí vận chuyển hai chiều.
[3] Hàng giảm giá trên 50% không áp dụng đổi trả.
</nguon>

Câu hỏi: Tôi đổi ý thì có được trả hàng không?
([], 0) ← Bạn được đổi trả miễn phí trong 7 ngày nếu hàng lỗi [1]. Nếu chỉ đổi ý, bạn trả phí ship hai chiều [2].
([4], 0) ← Bạn được đổi trả trong 30 ngày [4].
([], 1) ← Hàng giảm giá sâu không được đổi [3]. Shop luôn hoàn tiền trong 24 giờ.
